# Holographic Studio

Notebook điều khiển ứng dụng. Lần đầu trên máy mới, **chạy `setup.cmd`** (bấm đúp, hoặc
**Terminal → Run Task… → holostudio: cài đặt máy mới**) để cài Python, thư viện, model và
**kernel "Holographic Studio (.venv)"**. Sau đó chọn kernel đó ở góc trên bên phải và chạy từng ô.

In [ ]:
# 1. Kiểm tra môi trường: đúng kernel, đủ thư viện, đủ model.
import json
import os
import subprocess
import sys
from pathlib import Path

HERE = Path.cwd()
DESKTOP = HERE if (HERE / "holostudio").exists() else HERE / "desktop"
sys.path.insert(0, str(DESKTOP))
os.chdir(DESKTOP)

# Notebook test mode: skip the long steps (used by the automatic notebook check).
DRY_RUN = os.environ.get("HOLOSTUDIO_NOTEBOOK_DRY") == "1"
UTF8 = {**os.environ, "PYTHONIOENCODING": "utf-8"}

print("Python :", sys.version.split()[0], "-", sys.executable)
assert sys.version_info[:2] == (3, 13), "Hãy chọn kernel 'Holographic Studio (.venv)'"

import cv2  # noqa: E402
import mediapipe  # noqa: E402
import numpy  # noqa: E402

from holostudio import config  # noqa: E402

print("Thư viện: mediapipe", mediapipe.__version__, "| opencv", cv2.__version__)
print("          numpy", numpy.__version__)
for model in (config.HAND_LANDMARKER_FILE, config.POSE_LANDMARKER_FILE):
    state = f"{model.stat().st_size // 1024} KB" if model.exists() else "chưa tải (ô 2)"
    print(f"Model  : {model.name} - {state}")

In [ ]:
# 2. Tải model (chỉ lần đầu, sau đó dùng file đã lưu trong desktop/data/models).
from holostudio.identity.bodies import ensure_pose_model
from holostudio.tracking.model import ensure_hand_model

for path in (ensure_hand_model(), ensure_pose_model()):
    print("OK", path.name)

In [ ]:
# 3. Gate: lint + toàn bộ test (khoảng 10 giây).
def run(*args):
    command = [sys.executable, *args]
    result = subprocess.run(
        command, capture_output=True, text=True, encoding="utf-8", errors="replace", env=UTF8
    )
    for line in (result.stdout + result.stderr).strip().splitlines()[-3:]:
        print(line)
    return result.returncode


if DRY_RUN:
    print("(bỏ qua trong chế độ kiểm tra notebook)")
else:
    assert run("-m", "ruff", "check", ".") == 0, "ruff báo lỗi"
    assert run("-m", "pytest", "-q", "-p", "no:cacheprovider") == 0, "có test thất bại"
    print("Gate: ĐẠT")

In [ ]:
# 4. Chạy thử toàn bộ ứng dụng không cần cửa sổ, với bàn tay ảo theo kịch bản.
if DRY_RUN:
    print("(bỏ qua trong chế độ kiểm tra notebook)")
else:
    command = [sys.executable, "-m", "holostudio.app.smoke"]
    result = subprocess.run(command, capture_output=True, text=True, encoding="utf-8", env=UTF8)
    report = json.loads(result.stdout[result.stdout.index("{") :])
    for name, ok in report["checks"].items():
        print("ĐẠT " if ok else "LỖI ", name)
    print("Mỗi khung hình:", report["frame_ms_median"], "ms (95%:", report["frame_ms_p95"], "ms)")

In [ ]:
# 5. Mở ứng dụng trong cửa sổ riêng (đóng cửa sổ để thoát).
if DRY_RUN:
    print("(bỏ qua trong chế độ kiểm tra notebook)")
else:
    app = subprocess.Popen([sys.executable, "run.py"], cwd=DESKTOP)
    print("Đã mở Holographic Studio, PID", app.pid)